In [0]:
# CELL 1 — Imports

%pip install lightgbm scikit-learn pandas numpy mlflow

import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from mlflow.tracking import MlflowClient
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.getOrCreate()
client = MlflowClient()
mlflow.set_experiment("/Users/rajath2010rrp@gmail.com/hackbricks_credit_risk")

print("Imports done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ad63cc91-ec31-46c4-971c-3f71b8e9b474
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nf

In [0]:
# CELL 2 — Check PSI results from Delta table

print("Reading drift metrics from Delta table...")
drift_df = spark.sql("""
    SELECT feature_name, psi_value, drift_detected, computed_at
    FROM hackbricks.drift_metrics
    ORDER BY computed_at DESC
""").toPandas()

print("Latest drift metrics:")
display(drift_df)

# Check if retraining is needed
drifted_features = drift_df[drift_df['drift_detected'] == True]['feature_name'].tolist()
max_psi = drift_df['psi_value'].max()

print(f"\nMax PSI across all features: {max_psi:.4f}")
print(f"Features with drift: {drifted_features}")

RETRAIN_THRESHOLD = 0.2

if max_psi < RETRAIN_THRESHOLD:
    print(f"\nAll PSI values below {RETRAIN_THRESHOLD}. No retraining needed.")
    print("Champion model is still valid. Stopping here.")
    dbutils.notebook.exit("NO_DRIFT_DETECTED")
    # This line stops the notebook from running further cells
else:
    print(f"\nDrift detected! Max PSI = {max_psi:.4f} > {RETRAIN_THRESHOLD}")
    print("Proceeding with automated retraining...")


Reading drift metrics from Delta table...
Latest drift metrics:


feature_name,psi_value,drift_detected,computed_at
existing_credits,0.0,false,2026-04-11T14:14:31.829596
installment_rate,8.2832,true,2026-04-11T14:14:31.829592
duration_months,0.0,false,2026-04-11T14:14:31.829587
credit_amount,0.0,false,2026-04-11T14:14:31.829582
age,0.0,false,2026-04-11T14:14:31.829568



Max PSI across all features: 8.2832
Features with drift: ['installment_rate']

Drift detected! Max PSI = 8.2832 > 0.2
Proceeding with automated retraining...


In [0]:
# CELL 3 — Load and prepare data for retraining

train_df = pd.read_csv("/Volumes/workspace/default/hackbricks/application_train.csv")

TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"

# Same cleaning as Notebook 1
threshold = 0.6
cols_to_drop = [col for col in train_df.columns if train_df[col].isnull().mean() > threshold]
train_df = train_df.drop(columns=cols_to_drop)

y = train_df[TARGET_COL]
X = train_df.drop(columns=[TARGET_COL, ID_COL])

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in categorical_cols:
    X[col] = X[col].fillna("MISSING")
    X[col] = le.fit_transform(X[col].astype(str))
X = X.fillna(X.median())

# Check if the saved test set exists, otherwise create one from the data
try:
    test_set_spark = spark.sql("SELECT * FROM hackbricks.champion_test_set")
    test_df = test_set_spark.toPandas()
    print("Loaded saved test set from hackbricks.champion_test_set")
except:
    # Test set doesn't exist - create one from the data
    print("Test set not found. Creating test split from available data...")
    X_temp, X_test_full, y_temp, y_test_full = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    # Use the temp split for training, keep test separate
    X = X_temp
    y = y_temp
    test_df = X_test_full.copy()
    test_df['TARGET'] = y_test_full.values
    print(f"Created test set with {len(test_df):,} samples")

y_test = test_df['TARGET']
X_test = test_df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')

# Align columns — test set might have slightly different columns
common_cols = [c for c in X.columns if c in X_test.columns]
X = X[common_cols]
X_test = X_test[common_cols]

# Train/val split for retraining
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=99,  # Different seed = different split = slightly different model
    stratify=y
)

print(f"Retraining data: {len(X_train):,} train, {len(X_val):,} val, {len(X_test):,} test")

{"ts": "2026-04-11 14:24:10.956", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[TABLE_OR_VIEW_NOT_FOUND] The table or view `hackbricks`.`champion_test_set` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 14;\n'Project [*]\n+- 'UnresolvedRelation [hackbricks, champion_test_set], [], false\n\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[TABLE_OR_VIEW_NOT_FOUND] The table or view `hackbricks`.`champion_test_set` cannot be found. Verify the spelling and correctness 

Test set not found. Creating test split from available data...
Created test set with 61,503 samples
Retraining data: 196,806 train, 49,202 val, 61,503 test


In [0]:
# CELL 4 — Train Challenger model

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

CHALLENGER_PARAMS = {
    "n_estimators": 1000,         # More trees than Champion (was 500)
    "learning_rate": 0.02,        # Slower learning rate → often more accurate
    "num_leaves": 127,            # More complex trees (was 63)
    "max_depth": -1,
    "min_child_samples": 30,      # Lower than Champion (was 50)
    "subsample": 0.85,
    "colsample_bytree": 0.75,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 99,
    "n_jobs": -1,
    "verbose": -1,
    "reg_alpha": 0.1,             # L1 regularization (new — Champion didn't have this)
    "reg_lambda": 0.1             # L2 regularization (new — Champion didn't have this)
}

print("Training Challenger model...")
print("This may take 5-10 minutes with 1000 trees.\n")

with mlflow.start_run(run_name="challenger_retrain_post_drift") as run:

    CHALLENGER_RUN_ID = run.info.run_id
    print(f"Challenger Run ID: {CHALLENGER_RUN_ID}")

    mlflow.log_params(CHALLENGER_PARAMS)
    mlflow.log_param("trigger_reason", f"PSI drift detected: max_psi={max_psi:.4f}")
    mlflow.log_param("drifted_features", str(drifted_features))
    mlflow.log_param("model_type", "LightGBM_Challenger")

    # Train
    challenger_model = lgb.LGBMClassifier(**CHALLENGER_PARAMS)
    challenger_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=100)
        ]
    )

    # Evaluate
    challenger_test_preds = challenger_model.predict_proba(X_test)[:, 1]
    challenger_test_auc   = roc_auc_score(y_test, challenger_test_preds)
    challenger_gini       = 2 * challenger_test_auc - 1

    mlflow.log_metric("test_auc",       challenger_test_auc)
    mlflow.log_metric("gini_coefficient", challenger_gini)
    mlflow.log_metric("best_iteration", challenger_model.best_iteration_)

    # Generate signature for Unity Catalog
    signature = mlflow.models.infer_signature(
        X_train,
        challenger_model.predict_proba(X_train)
    )

    # Register as Challenger
    mlflow.lightgbm.log_model(
        challenger_model,
        artifact_path="model",
        signature=signature,
        input_example=X_train[:5],
        registered_model_name="workspace.default.credit-risk-champion"
        # Same model name as Champion — this creates a new VERSION
        # MLflow auto-increments: Champion was v1, Challenger will be v2
    )

    print(f"\nChallenger Test AUC:  {challenger_test_auc:.4f}")
    print(f"Challenger Gini:      {challenger_gini:.4f}")

print("\nChallenger training complete!")

Training Challenger model...
This may take 5-10 minutes with 1000 trees.

Challenger Run ID: aae0871da30e44e5b543b375f4524571
Training until validation scores don't improve for 100 rounds
[100]	valid_0's binary_logloss: 0.490465
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.275939


2026/04/11 14:26:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3f31c628-f679.cloud.databricks.com/ml/experiments/2608798312767667/models/m-a7b4d0cbffea4f6a98ca54b58c3d8e6d?o=7474656095211125
Registered model 'workspace.default.credit-risk-champion' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]


Challenger Test AUC:  0.7312
Challenger Gini:      0.4624

Challenger training complete!


🔗 Created version '3' of model 'workspace.default.credit-risk-champion': https://dbc-3f31c628-f679.cloud.databricks.com/explore/data/models/workspace/default/credit-risk-champion/version/3?o=7474656095211125


In [0]:
# CELL 5 — Get Champion metrics to compare

champion_model_info = client.get_model_version_by_alias("credit-risk-champion", "Champion")
champion_version    = champion_model_info.version
champion_run_id     = champion_model_info.run_id

# Get Champion's metrics from its MLflow run
champion_run = client.get_run(champion_run_id)
champion_test_auc = champion_run.data.metrics.get("test_auc", 0)
champion_gini     = champion_run.data.metrics.get("gini_coefficient", 0)

print(f"Champion (v{champion_version}) metrics:")
print(f"  Test AUC:  {champion_test_auc:.4f}")
print(f"  Gini:      {champion_gini:.4f}")

print(f"\nChallenger metrics:")
print(f"  Test AUC:  {challenger_test_auc:.4f}")
print(f"  Gini:      {challenger_gini:.4f}")

Champion (v2) metrics:
  Test AUC:  0.7231
  Gini:      0.4463

Challenger metrics:
  Test AUC:  0.7312
  Gini:      0.4624


In [0]:
# CELL 6 — Champion vs Challenger comparison using mlflow.evaluate()
import mlflow.pyfunc

print("\nRunning mlflow.evaluate() for Champion vs Challenger comparison...\n")

# Prepare eval dataset in the format mlflow.evaluate() expects
eval_data = X_test.copy()
eval_data["label"] = y_test.values
eval_spark_df = spark.createDataFrame(eval_data)
eval_spark_df.write.format("delta").mode("overwrite").saveAsTable("hackbricks.eval_dataset")

# Load the eval dataset as a pandas dataframe for mlflow.evaluate()
eval_dataset = mlflow.data.from_pandas(
    eval_data,
    targets="label"
)

with mlflow.start_run(run_name="champion_vs_challenger_eval") as compare_run:

    # --- Evaluate Champion ---
    champion_uri = f"models:/workspace.default.credit-risk-champion@Champion"

    champion_eval_result = mlflow.evaluate(
        model=champion_uri,
        data=eval_dataset,
        model_type="classifier",
        evaluators="default"
    )

    # --- Get Challenger version number ---
    all_versions = client.search_model_versions(f"name='workspace.default.credit-risk-champion'")
    challenger_version_num = max(int(v.version) for v in all_versions)
    challenger_uri = f"models:/workspace.default.credit-risk-champion/{challenger_version_num}"

    challenger_eval_result = mlflow.evaluate(
        model=challenger_uri,
        data=eval_dataset,
        model_type="classifier",
        evaluators="default"
    )

    # Log comparison metrics
    champion_auc_eval    = champion_eval_result.metrics.get("roc_auc", champion_test_auc)
    challenger_auc_eval  = challenger_eval_result.metrics.get("roc_auc", challenger_test_auc)

    mlflow.log_metric("champion_auc",    champion_auc_eval)
    mlflow.log_metric("challenger_auc",  challenger_auc_eval)
    mlflow.log_metric("auc_improvement", challenger_auc_eval - champion_auc_eval)

    print("=== Champion vs Challenger Scorecard ===")
    print(f"{'Metric':<25} {'Champion':>12} {'Challenger':>12} {'Winner':>10}")
    print("-" * 65)
    print(f"{'AUC-ROC':<25} {champion_auc_eval:>12.4f} {challenger_auc_eval:>12.4f} {'Challenger' if challenger_auc_eval > champion_auc_eval else 'Champion':>10}")
    print(f"{'Gini Coefficient':<25} {champion_gini:>12.4f} {challenger_gini:>12.4f} {'Challenger' if challenger_gini > champion_gini else 'Champion':>10}")

    # Decision
    if challenger_auc_eval > champion_auc_eval:
        winner = "CHALLENGER"
        mlflow.set_tag("winner", "CHALLENGER")
        mlflow.set_tag("promotion_action", "Challenger promoted to Champion")
    else:
        winner = "CHAMPION"
        mlflow.set_tag("winner", "CHAMPION")
        mlflow.set_tag("promotion_action", "Champion retained")

    print(f"\nWINNER: {winner}")


Running mlflow.evaluate() for Champion vs Challenger comparison...



2026/04/11 14:47:15 WARNING mlflow.models.evaluation.evaluators.classifier: According to the evaluation dataset label values, the model type looks like None, but you specified model type 'classifier'. Please verify that you set the `model_type` and `dataset` arguments correctly.
2026/04/11 14:47:16 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.
2026/04/11 14:47:16 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2026/04/11 14:47:16 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


2026/04/11 14:47:21 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-a7b4d0cbffea4f6a98ca54b58c3d8e6d
2026/04/11 14:47:21 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/04/11 14:47:21 WARNING mlflow.models.evaluation.evaluators.classifier: According to the evaluation dataset label values, the model type looks like None, but you specified model type 'classifier'. Please verify that you set the `model_type` and `dataset` arguments correctly.
2026/04/11 14:47:21 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.
2026/04/11 14:47:21 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2026/04/11 14:47:22 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


=== Champion vs Challenger Scorecard ===
Metric                        Champion   Challenger     Winner
-----------------------------------------------------------------
AUC-ROC                         0.7231       0.7312 Challenger
Gini Coefficient                0.4463       0.4624 Challenger

WINNER: CHALLENGER


<Figure size 1050x700 with 0 Axes>

In [0]:
# -----------------------------------------------------------------------------
# CELL 7 — Promote Challenger to Champion (if it won)
# -----------------------------------------------------------------------------

if winner == "CHALLENGER":
    print(f"\nPromoting Challenger (v{challenger_version_num}) to Champion...")

    # Remove Champion alias from old version
    old_champion_version = champion_version
    client.delete_registered_model_alias("credit-risk-champion", "Champion")

    # Set Champion alias on new version (Challenger)
    client.set_registered_model_alias(
        name="credit-risk-champion",
        alias="Champion",
        version=str(challenger_version_num)
    )

    # Archive old Champion
    client.set_registered_model_alias(
        name="credit-risk-champion",
        alias="Retired_Champion_v" + str(old_champion_version),
        version=str(old_champion_version)
    )

    # Add description
    client.update_model_version(
        name="credit-risk-champion",
        version=str(challenger_version_num),
        description=f"Promoted Challenger. Test AUC: {challenger_auc_eval:.4f}. Replaced Champion v{old_champion_version} (AUC: {champion_auc_eval:.4f}). Triggered by PSI drift."
    )

    print(f"Challenger v{challenger_version_num} is now the Champion!")
    print(f"Old Champion v{old_champion_version} has been archived.")
else:
    print(f"\nChampion retained. Challenger (v{challenger_version_num}) did not improve.")
    print("No model promotion performed.")



Promoting Challenger (v3) to Champion...
Challenger v3 is now the Champion!
Old Champion v2 has been archived.


In [0]:
# -----------------------------------------------------------------------------
# CELL 8 — Log comparison to Delta table (Governance)
# -----------------------------------------------------------------------------
from datetime import datetime

comparison_record = {
    'run_timestamp':        datetime.now().isoformat(),
    'trigger_reason':       'psi_drift',
    'max_psi':              float(max_psi),
    'drifted_features':     str(drifted_features),
    'champion_version':     int(champion_version),
    'challenger_version':   int(challenger_version_num),
    'champion_auc':         float(champion_auc_eval),
    'challenger_auc':       float(challenger_auc_eval),
    'champion_gini':        float(champion_gini),
    'challenger_gini':      float(challenger_gini),
    'auc_improvement':      float(challenger_auc_eval - champion_auc_eval),
    'winner':               winner,
    'promoted':             (winner == "CHALLENGER")
}

comparison_df = pd.DataFrame([comparison_record])
spark.createDataFrame(comparison_df) \
    .write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("hackbricks.retraining_logs")

print("\nRetraining log saved to Delta table: hackbricks.retraining_logs")
display(spark.sql("SELECT * FROM hackbricks.retraining_logs ORDER BY run_timestamp DESC LIMIT 5"))
print("\nNotebook 3 complete. Proceed to Notebook 4 (LLM Eval).")



Retraining log saved to Delta table: hackbricks.retraining_logs


run_timestamp,trigger_reason,max_psi,drifted_features,champion_version,challenger_version,champion_auc,challenger_auc,champion_gini,challenger_gini,auc_improvement,winner,promoted
2026-04-11T14:47:59.700969,psi_drift,8.2832,['installment_rate'],2,3,0.7231394529494536,0.7311866179033774,0.4462789058989072,0.4623732358067547,0.008047164953923769,CHALLENGER,true



Notebook 3 complete. Proceed to Notebook 4 (LLM Eval).


In [0]:
# Run this in any Databricks notebook
# Creates a secret scope called "hackbricks-secrets"

# Check if scope already exists
existing_scopes = [s.name for s in dbutils.secrets.listScopes()]
print("Existing scopes:", existing_scopes)

if "hackbricks-secrets" not in existing_scopes:
    # Create via Databricks CLI command
    # Run this in a notebook cell
    import subprocess
    result = subprocess.run([
        "databricks", "secrets", "create-scope",
        "--scope", "hackbricks-secrets"
    ], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    print("Scope already exists")

Existing scopes: []
The Databricks CLI is only supported for interactive use from the web terminal. We recommend using the web terminal to invoke CLI commands. If you would like to interface with Databricks APIs then we recommend using the Databricks Python SDK.


